# High-z Example 20: Beam Smearing Assessment

**EPS Research — High-z Kinematic Corpus Z1**

All ALPINE data has ~1 arcsec beam (~6-7 kpc at z~5).
This sets the spatial resolution floor and affects inner-ring
kinematic measurements. 3DBarolo mitigates but does not
eliminate beam-smearing effects.

This example quantifies the beam size relative to galaxy size.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.21834678  
**arXiv:** 2605.25339  
**Source:** Jones et al. (2021), MNRAS 507, 3540; Le Fevre et al. (2020)  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'high_z_kinematic_corpus_Z1.json': 'https://zenodo.org/records/21834678/files/high_z_kinematic_corpus_Z1.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    if not os.path.exists("high_z_kinematic_corpus_Z1_flat.csv"):
        import json, csv
        _g = json.load(open("high_z_kinematic_corpus_Z1.json"))["galaxies"]
        _k = [k for k in _g[0] if not isinstance(_g[0][k], (dict, list))]
        with open("high_z_kinematic_corpus_Z1_flat.csv", "w", newline="") as _f:
            _w = csv.DictWriter(_f, fieldnames=_k)
            _w.writeheader()
            for _r in _g:
                _w.writerow({k: ("" if _r.get(k) is None else _r.get(k)) for k in _k})
        print("  built high_z_kinematic_corpus_Z1_flat.csv from the JSON (the Zenodo record has no flat CSV)")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('high_z_kinematic_corpus_Z1.json') as f:
    corpus = json.load(f)

galaxies  = corpus['galaxies']
rotators  = [g for g in galaxies if g.get('is_rotator') and g.get('quality_tier')==1]
print(f"Total galaxies: {len(galaxies)}")
print(f"Tier-1 rotators: {len(rotators)}")


In [ ]:
import numpy as np
# ALMA beam: ~1 arcsec FWHM
# Angular diameter distance at z~5: ~1740 Mpc (flat LCDM)
# 1 arcsec at z=5 ~ 6.3 kpc

beam_kpc = 6.3  # approximate beam size in kpc at z~5

print(f"ALMA beam size: ~1 arcsec FWHM")
print(f"Physical scale: ~{beam_kpc:.1f} kpc at z~5")
print()
print("Beam size relative to rotation curve extent:")
print(f"{'Galaxy':<20} {'z':>6} {'R_max':>7} {'Beam/R_max':>11} {'N_rings':>8}")
print('-'*58)

for g in sorted(rotators, key=lambda x: x['redshift']):
    R_max   = g['data'][-1]['R_kpc']
    n_rings = len(g['data'])
    ratio   = beam_kpc / R_max
    flag    = ' ← marginal' if ratio > 0.4 else ''
    print(f"{g['galaxy']:<20} {g['redshift']:>6.4f} {R_max:>7.2f} "
          f"{ratio:>11.2f}{flag}  {n_rings:>8}")

print(f"\nNote: Beam/R_max > 0.4 means beam spans >40% of galaxy — poorly resolved")